# House Price Prediction: train / validation / test pipeline


Источник идеи: GeeksforGeeks, "House Price Prediction using Machine Learning in Python".
URL: https://www.geeksforgeeks.org/machine-learning/house-price-prediction-using-machine-learning-in-python/

Этот notebook не является дословной копией статьи. Он повторяет учебную логику статьи, но переписан как подробный tutorial для новичка: перед кодом есть теория, а строки кода прокомментированы.


## Чем этот notebook отличается от baseline

Первый notebook повторяет простую учебную схему: подготовили таблицу, сделали encoding, затем split 80/20.

Здесь мы сделаем более правильный data science pipeline:

1. Сначала отделим `test` set и не будем смотреть на него до финальной оценки.
2. Из оставшихся данных сделаем `train` и `validation`.
3. Preprocessing будет обучаться только на `train`.
4. Validation будем использовать для выбора модели.
5. Test используем один раз в конце для честной финальной оценки.

Это защищает от data leakage и ближе к реальной работе data scientist.

## 1. Импорт библиотек

В этом notebook мы используем `Pipeline` и `ColumnTransformer`.

Зачем они нужны:

- `Pipeline` соединяет preprocessing и модель в один объект.
- `ColumnTransformer` позволяет по-разному обрабатывать числовые и категориальные колонки.
- `SimpleImputer` заполняет пропуски.
- `OneHotEncoder` кодирует категории.

Главная польза: preprocessing учится только на train set, а потом применяется к validation/test. Это правильнее, чем обучать encoder на всей таблице заранее.

In [ ]:
# pathlib.Path дает удобный объект для путей к файлам и папкам.
from pathlib import Path

# urllib.request.urlopen умеет скачивать файл по URL.
from urllib.request import urlopen, Request

# ssl нужен, чтобы явно управлять проверкой сертификатов при скачивании учебного файла.
import ssl

# pandas нужен для табличных данных.
import pandas as pd

# numpy нужен для математических операций.
import numpy as np

# matplotlib нужен для графиков.
import matplotlib.pyplot as plt

# seaborn нужен для красивых статистических графиков.
import seaborn as sns

# train_test_split делит данные на части.
from sklearn.model_selection import train_test_split

# ColumnTransformer применяет разные preprocessing шаги к разным колонкам.
from sklearn.compose import ColumnTransformer

# Pipeline объединяет preprocessing и модель.
from sklearn.pipeline import Pipeline

# SimpleImputer заполняет missing values.
from sklearn.impute import SimpleImputer

# OneHotEncoder превращает текстовые категории в числовые признаки.
from sklearn.preprocessing import OneHotEncoder

# StandardScaler масштабирует числовые признаки; это особенно важно для SVR.
from sklearn.preprocessing import StandardScaler

# Регрессионные модели.
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge

# Метрики качества регрессии.
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error, r2_score

# Настраиваем стиль графиков.
sns.set_theme(style="whitegrid")

# Фиксируем random_state для воспроизводимости.
RANDOM_STATE = 42

## 2. Загрузка данных

Скачиваем тот же Excel dataset, что и в baseline notebook.

Важно: после загрузки мы сразу отделим target `SalePrice` от признаков.

In [ ]:
# Прямая ссылка на Excel dataset из статьи GeeksforGeeks.
DATA_URL = "https://media.geeksforgeeks.org/wp-content/uploads/20260116164015462273/HousePricePrediction.xlsx"

# Папка для хранения dataset.
DATA_DIR = Path("../data/house_price")

# Создаем папку, если ее еще нет.
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Путь к локальному Excel-файлу.
DATA_PATH = DATA_DIR / "HousePricePrediction.xlsx"

# Если файла нет, скачиваем его.
if not DATA_PATH.exists():
    # Создаем HTTP-запрос с User-Agent, чтобы сервер понял, что это обычный учебный download.
    request = Request(DATA_URL, headers={"User-Agent": "Mozilla/5.0"})
    
    # Создаем SSL context без строгой проверки сертификата, потому что на некоторых локальных машинах
    # учебный download может падать из-за корпоративного или системного certificate chain.
    ssl_context = ssl._create_unverified_context()
    
    # Открываем URL и читаем байты Excel-файла.
    with urlopen(request, context=ssl_context, timeout=30) as response:
        file_bytes = response.read()
    
    # Записываем скачанные байты в локальный .xlsx файл.
    DATA_PATH.write_bytes(file_bytes)

# Загружаем Excel в DataFrame.
dataset = pd.read_excel(DATA_PATH)

# Показываем размер таблицы.
print("Dataset shape:", dataset.shape)

# Показываем первые строки.
dataset.head()

## 3. Мини-EDA перед split

До split можно делать только общий безопасный осмотр:

- размер таблицы;
- названия колонок;
- типы данных;
- количество пропусков.

Но нельзя принимать решения, которые используют информацию из test set для настройки модели. Например, encoder и imputer должны обучаться только на train.

In [ ]:
# Показываем типы колонок.
dataset.dtypes

In [ ]:
# Считаем пропуски в каждой колонке и сортируем по убыванию.
missing_summary = dataset.isnull().sum().sort_values(ascending=False)

# Показываем только колонки, где есть хотя бы один пропуск.
missing_summary[missing_summary > 0]

## 4. Отделяем target и удаляем Id

`SalePrice` - это target, то есть то, что мы предсказываем.

`Id` - это идентификатор записи. Обычно он не несет полезного экономического смысла для prediction, поэтому его удаляем.

Важный момент: строки с отсутствующим `SalePrice` нельзя использовать для supervised training, потому что у них нет правильного ответа. В статье `SalePrice` заполняется средним, но для строгой ML-постановки лучше удалить строки без target.

In [ ]:
# Создаем копию исходной таблицы.
model_data = dataset.copy()

# Удаляем Id, если он есть.
if "Id" in model_data.columns:
    model_data = model_data.drop("Id", axis=1)

# Удаляем строки без target, потому что модель не может учиться без правильной цены.
model_data = model_data.dropna(subset=["SalePrice"])

# X содержит все признаки, кроме target.
X = model_data.drop("SalePrice", axis=1)

# y содержит только target.
y = model_data["SalePrice"]

# Проверяем размеры.
print("X shape:", X.shape)
print("y shape:", y.shape)

## 5. Разделение на train, validation и test

Мы делаем три части:

- `train`: модель учится на этих данных;
- `validation`: выбираем лучшую модель и настройки;
- `test`: финальная честная проверка один раз в конце.

Пропорции:

```text
train = 60%
validation = 20%
test = 20%
```

Сначала отделяем test 20%, потом оставшиеся 80% делим на train и validation.

In [ ]:
# Первый split: отделяем test set, который не будем использовать при выборе модели.
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

# Второй split: из оставшихся 80% делаем train 60% и validation 20% от всего dataset.
X_train, X_valid, y_train, y_valid = train_test_split(
    X_temp,
    y_temp,
    test_size=0.25,  # 0.25 от 80% = 20% от всего dataset
    random_state=RANDOM_STATE
)

# Печатаем размеры всех частей.
print("Train:     ", X_train.shape, y_train.shape)
print("Validation:", X_valid.shape, y_valid.shape)
print("Test:      ", X_test.shape, y_test.shape)

# Проверяем доли.
total_rows = len(X)
print("Train fraction:     ", round(len(X_train) / total_rows, 2))
print("Validation fraction:", round(len(X_valid) / total_rows, 2))
print("Test fraction:      ", round(len(X_test) / total_rows, 2))

## 6. Создаем preprocessing pipeline

У нас есть два типа признаков:

1. Числовые признаки: можно заполнить медианой и масштабировать.
2. Категориальные признаки: можно заполнить самым частым значением и применить One-Hot Encoding.

Почему это делается через pipeline:

- imputer запоминает медиану/моду только на train;
- encoder запоминает категории только на train;
- validation и test не влияют на preprocessing;
- это снижает риск data leakage.

In [ ]:
# Находим числовые колонки в X_train.
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()

# Находим категориальные колонки в X_train.
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

# Печатаем найденные группы признаков.
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

# Pipeline для числовых признаков.
numeric_transformer = Pipeline(steps=[
    # SimpleImputer(strategy='median') заменяет пропуски медианой train колонки.
    ("imputer", SimpleImputer(strategy="median")),
    
    # StandardScaler приводит числовые признаки к похожему масштабу.
    ("scaler", StandardScaler()),
])

# Pipeline для категориальных признаков.
categorical_transformer = Pipeline(steps=[
    # Заполняем пропуски самым частым значением категории.
    ("imputer", SimpleImputer(strategy="most_frequent")),
    
    # OneHotEncoder создает dummy columns; unknown категории на valid/test игнорируются.
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

# ColumnTransformer применяет нужный transformer к нужным колонкам.
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

## 7. Обучаем несколько моделей через Pipeline

Каждая модель будет внутри своего Pipeline:

```text
raw X -> preprocessing -> model -> prediction
```

Это значит, что мы передаем в `.fit()` исходные признаки, а pipeline сам делает имputation, scaling, encoding и обучение модели.

In [ ]:
# Словарь candidate models: имя -> модель.
candidate_models = {
    "SVR": SVR(),
    "RandomForestRegressor": RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE),
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0, random_state=RANDOM_STATE),
}

# Сюда будем складывать validation метрики.
validation_results = []

# Сюда сохраним обученные pipelines, чтобы потом взять лучшую модель.
fitted_pipelines = {}

# Обучаем каждую модель.
for model_name, model in candidate_models.items():
    # Создаем pipeline из preprocessing и модели.
    pipeline = Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", model),
    ])
    
    # Обучаем pipeline только на train данных.
    pipeline.fit(X_train, y_train)
    
    # Делаем prediction на validation set.
    valid_pred = pipeline.predict(X_valid)
    
    # Считаем validation metrics.
    mape = mean_absolute_percentage_error(y_valid, valid_pred)
    mae = mean_absolute_error(y_valid, valid_pred)
    rmse = np.sqrt(mean_squared_error(y_valid, valid_pred))
    r2 = r2_score(y_valid, valid_pred)
    
    # Сохраняем результаты.
    validation_results.append({
        "model": model_name,
        "valid_MAPE": mape,
        "valid_MAE": mae,
        "valid_RMSE": rmse,
        "valid_R2": r2,
    })
    
    # Сохраняем обученный pipeline.
    fitted_pipelines[model_name] = pipeline

# Создаем таблицу результатов и сортируем по MAPE.
validation_results_df = pd.DataFrame(validation_results).sort_values("valid_MAPE")

# Показываем validation leaderboard.
validation_results_df

## 8. Выбираем лучшую модель по validation

Validation set нужен именно для выбора модели.

Мы выбираем модель с минимальным `valid_MAPE`, потому что MAPE легко интерпретировать как среднюю относительную ошибку.

После выбора модели мы наконец оцениваем ее на test set.

In [ ]:
# Берем имя лучшей модели из первой строки отсортированной таблицы.
best_model_name = validation_results_df.iloc[0]["model"]

# Достаем обученный pipeline лучшей модели.
best_pipeline = fitted_pipelines[best_model_name]

# Делаем prediction на test set, который не использовался при выборе модели.
test_pred = best_pipeline.predict(X_test)

# Считаем test metrics.
test_metrics = {
    "model": best_model_name,
    "test_MAPE": mean_absolute_percentage_error(y_test, test_pred),
    "test_MAE": mean_absolute_error(y_test, test_pred),
    "test_RMSE": np.sqrt(mean_squared_error(y_test, test_pred)),
    "test_R2": r2_score(y_test, test_pred),
}

# Показываем выбранную модель и test metrics.
print("Best model selected on validation:", best_model_name)
pd.DataFrame([test_metrics])

## 9. График actual vs predicted на test set

Теперь график строится именно на test set.

Это честнее, потому что test не участвовал в выборе модели.

In [ ]:
# Создаем график.
plt.figure(figsize=(7, 7))

# Рисуем реальные test цены по оси x и predicted цены по оси y.
sns.scatterplot(x=y_test, y=test_pred, alpha=0.6)

# Считаем диапазон для диагональной линии.
min_price = min(y_test.min(), test_pred.min())
max_price = max(y_test.max(), test_pred.max())

# Рисуем красную пунктирную линию идеального prediction.
plt.plot([min_price, max_price], [min_price, max_price], color="red", linestyle="--")

# Добавляем подписи.
plt.title(f"Test actual vs predicted: {best_model_name}")
plt.xlabel("Actual SalePrice")
plt.ylabel("Predicted SalePrice")

# Делаем layout аккуратнее.
plt.tight_layout()

# Показываем график.
plt.show()

## 10. Сравнение: почему train/validation/test лучше

Baseline схема 80/20 полезна для первого знакомства, но у нее есть ограничение: если preprocessing сделать до split, часть информации из validation может попасть в encoder/imputer.

Более строгая схема:

1. Сначала split.
2. Preprocessing `.fit()` только на train.
3. Validation для выбора модели.
4. Test только для финальной оценки.

Такой подход лучше имитирует реальную ситуацию, где будущие данные заранее неизвестны.

In [ ]:
# Создаем короткую таблицу, которая показывает роль каждого split.
split_roles = pd.DataFrame([
    {"split": "train", "used_for": "fit preprocessing and model parameters", "can_choose_model": "no"},
    {"split": "validation", "used_for": "compare candidate models", "can_choose_model": "yes"},
    {"split": "test", "used_for": "final unbiased evaluation", "can_choose_model": "no"},
])

# Показываем таблицу ролей.
split_roles

## 11. Итог train/validation/test notebook

Что мы сделали лучше, чем в простом baseline:

- отделили test set до выбора модели;
- не использовали test для настройки;
- обучили preprocessing только на train;
- сравнили модели на validation;
- лучшую модель оценили на test один раз;
- использовали `Pipeline` и `ColumnTransformer`, чтобы уменьшить риск data leakage.

Это более правильный data science workflow для регрессионной задачи house price prediction.